# 経済データ分析Ⅱ　第2回
# 住宅の広さと人口密度

石川県や富山県の住宅は、東京都と比べてどれほど広いのだろうか。持ち家と借家では、広さの違いも異なる。

全国47都道府県の公表値から、住宅の広さと人口密度の関係を調べる。経済データ分析Ⅰで学んだ単回帰を復習し、残差の意味を確認する。後半では、説明変数を二つ以上使う重回帰分析を学び、よく似た説明変数を同時に使うときの注意点を数値例で説明する。


## Notebook の使い方

コードセルを上から順に選び、`Shift + Enter` で実行します。各節の解説を読みながら、計算結果と図を確認してください。

この Notebook には47都道府県のデータが入っています。CSV ファイルを別に用意する必要はありません。JupyterLite では **Python (Pyodide)**、ローカルの Jupyter では Python カーネルを使います。実行結果を残すには Notebook を保存し、ブラウザで使った場合は `.ipynb` をダウンロードしてください。

図では `Owner-occupied` が持ち家、`Rented` が借家、`Population density` が人口密度、`Floor area` が延べ面積、`Residual` が残差を表します。`Ishikawa`・`Toyama`・`Tokyo` は石川県・富山県・東京都です。


## 1. データの定義と出典

出典は総務省統計局『統計でみる都道府県のすがた2026』である。**1行は1都道府県**で、住宅の数値は各県の平均である。

| 指標 | 対象年 | 単位 |
|---|---|---|
| 持ち家住宅の延べ面積（1住宅当たり） | 2023年 | m² |
| 借家住宅の延べ面積（1住宅当たり） | 2023年 | m² |
| 総面積1km²当たり人口密度 | 2024年 | 人/km² |
| 可住地面積1km²当たり人口密度 | 2024年 | 人/km² |

持ち家は、居住世帯が全部または一部を所有する住宅である。一戸建てに限らない。延べ面積は住宅の各階の床面積の合計で、敷地面積や1人当たりの面積とは異なる。住宅と人口密度は対象年が1年違う。

- [統計局：統計でみる都道府県のすがた2026](https://www.stat.go.jp/data/k-sugata/naiyou.html)
- [公式PDF](https://www.stat.go.jp/data/k-sugata/pdf/all_ken2026.pdf)：人口密度は冊子3頁、住宅面積は冊子60頁（PDFでは27・84頁）。指標コードは順に H0210301、H0210302、A01201、A01202。

47都道府県をすべて含むが、住宅面積の公表値は住宅・土地統計調査に基づく推計値である。県の平均どうしの比較から、個々の家計の関係がそのまま分かるわけではない。


In [ ]:
import sys
if sys.platform == 'emscripten':
    import pyodide_js
    await pyodide_js.loadPackage(['numpy', 'pandas', 'matplotlib', 'statsmodels', 'jinja2'])

from io import StringIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import statsmodels.api as sm
from IPython.display import display

pd.options.display.float_format = '{:.2f}'.format
plt.rcParams.update({
    'figure.figsize': (8, 4.5), 'figure.dpi': 110,
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.2,
})

labels = {'石川県': 'Ishikawa', '富山県': 'Toyama', '東京都': 'Tokyo'}
colors = {'石川県': '#a31b2a', '富山県': '#d38826', '東京都': '#26738d'}

def annotate_prefectures(ax, data, x_col, y_col):
    offsets = {'石川県': (7, -17), '富山県': (7, 7), '東京都': (-50, 8)}
    if y_col == 'log_inhabitable_density':
        offsets = {'石川県': (10, 10), '富山県': (-48, -22), '東京都': (-50, 8)}
    for name in labels:
        row = data.loc[data['prefecture'].eq(name)].iloc[0]
        ax.scatter(row[x_col], row[y_col], color=colors[name], s=55, zorder=4)
        ax.annotate(labels[name], (row[x_col], row[y_col]),
                    xytext=offsets[name], textcoords='offset points', fontsize=10)


In [ ]:
# 総務省統計局『統計でみる都道府県のすがた2026』の47都道府県の公表値
# 人口密度: 2024年 / 住宅面積: 2023年
data_csv = """prefecture_code,prefecture,population_density_per_km2,inhabitable_density_per_km2,owned_home_floor_area_m2,rented_home_floor_area_m2
1,北海道,64.3,222.2,120.1,50.9
2,青森県,120.8,358.2,145.2,52.5
3,岩手県,75.0,305.2,145.8,49.3
4,宮城県,308.7,705.6,127.9,45.4
5,秋田県,77.1,277.4,154.3,51.3
6,山形県,108.4,351.9,159.9,50.0
7,福島県,126.4,412.0,139.2,48.9
8,茨城県,460.1,721.4,126.9,47.5
9,栃木県,294.2,627.3,128.7,47.6
10,群馬県,297.1,833.0,128.7,48.8
11,埼玉県,1930.6,2816.7,106.4,44.4
12,千葉県,1212.3,1769.4,109.0,44.3
13,東京都,6444.7,9923.8,90.5,39.8
14,神奈川県,3817.4,6257.3,97.9,42.1
15,新潟県,166.8,461.3,155.7,48.3
16,富山県,234.7,541.2,167.5,49.0
17,石川県,262.3,787.3,151.8,47.7
18,福井県,176.3,686.0,163.5,51.9
19,山梨県,177.1,829.9,132.6,48.8
20,長野県,146.5,611.6,144.1,49.9
21,岐阜県,180.4,866.7,141.9,50.8
22,静岡県,453.5,1271.3,127.5,47.8
23,愛知県,1442.0,2490.1,123.7,46.5
24,三重県,296.3,828.9,131.1,49.5
25,滋賀県,349.0,1078.8,137.2,47.4
26,京都府,546.4,2140.5,110.5,41.5
27,大阪府,4596.0,6564.1,99.9,43.2
28,兵庫県,635.3,1927.2,114.2,47.6
29,奈良県,348.1,1504.9,127.8,53.6
30,和歌山県,186.3,783.4,123.8,52.1
31,鳥取県,151.4,587.2,147.2,51.1
32,島根県,95.7,505.2,151.1,51.4
33,岡山県,257.4,821.7,131.4,46.9
34,広島県,320.1,1181.4,119.5,46.0
35,山口県,209.6,746.7,124.6,50.5
36,徳島県,165.2,674.0,133.8,50.0
37,香川県,488.6,912.4,132.1,49.5
38,愛媛県,224.8,766.0,123.7,50.6
39,高知県,92.4,565.5,117.3,51.3
40,福岡県,1020.9,1841.9,114.8,45.9
41,佐賀県,322.9,590.5,138.9,53.7
42,長崎県,303.1,750.5,121.5,51.4
43,熊本県,229.0,617.8,125.0,50.1
44,大分県,171.1,604.4,123.5,47.6
45,宮崎県,133.6,550.9,115.2,51.0
46,鹿児島県,166.8,466.0,108.8,50.7
47,沖縄県,642.4,1301.8,106.0,48.8
"""

df = pd.read_csv(StringIO(data_csv))
print(f"都道府県数：{len(df)}")
df.head()


## 2. 石川・富山・東京の住宅面積

持ち家と借家を分けて比べる。表の値は「その県の住宅がすべてこの広さ」という意味ではなく、1住宅当たりの平均である。


In [ ]:
chosen = (df.set_index('prefecture')
          .loc[['石川県', '富山県', '東京都'],
               ['owned_home_floor_area_m2', 'rented_home_floor_area_m2']])
chosen.columns = ['持ち家（m²）', '借家（m²）']
display(chosen.style.format('{:.1f}'))

positions = np.arange(len(chosen))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(positions - 0.18, chosen['持ち家（m²）'], width=0.36,
       label='Owner-occupied', color='#a31b2a')
ax.bar(positions + 0.18, chosen['借家（m²）'], width=0.36,
       label='Rented', color='#26738d')
ax.set(xticks=positions, xticklabels=['Ishikawa', 'Toyama', 'Tokyo'],
       ylabel='Floor area per dwelling (m²)', ylim=(0, 190))
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.0f}'))
ax.legend()
plt.tight_layout()
plt.show()


### 住宅面積の違い

石川と東京の持ち家の差は $151.8-90.5=61.3$ m²、借家の差は $47.7-39.8=7.9$ m²である。持ち家では石川のほうが平均61.3 m²広く、借家よりも地域間の差が大きくなっている。富山の持ち家は平均167.5 m²で、この3都県では最も広くなっている。

住宅面積には人口密度のほか、世帯人数、戸建て住宅の割合、地価なども関係すると考えられる。この表は地域間の違いを示しているが、それぞれの要因がどの程度関係しているかまでは分からない。


## 3. 人口密度と住宅面積の散布図

人口密度が高い地域では、限られた土地に多くの人が住んでいる。住宅の広さとの関係を調べるため、横軸に総面積当たりの人口密度、縦軸に持ち家の床面積を置く。1点が1都道府県である。

図では、人口密度が高い都府県ほど持ち家の床面積が小さい傾向が見られる。ただし、多くの県が横軸の左側に集まるため、低密度の県どうしの違いは読み取りにくくなっている。


In [ ]:
fig, ax = plt.subplots()
ax.scatter(df['population_density_per_km2'], df['owned_home_floor_area_m2'],
           color='#777777', s=30, alpha=0.75)
annotate_prefectures(ax, df, 'population_density_per_km2', 'owned_home_floor_area_m2')
ax.set(xlabel='Population density (people/km²)',
       ylabel='Owner-occupied floor area (m²)')
ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.0f}'))
plt.tight_layout()
plt.show()


## 4. 人口密度の対数

元の尺度では、多くの県が図の左側に集まる。人口密度の自然対数を使うと、高密度の都府県との間隔を縮め、倍率の違いに注目できる。

$$x_i=\ln(人口密度_i).$$

$\ln$ は自然対数で、Python では `np.log()` である。人口密度が100から200になった場合も、1,000から2,000になった場合も、対数の差は $\ln 2$ になる。

$$\ln(2d)-\ln(d)=\ln 2\approx0.693.$$

対数を取ると関係の形が変わるため、元の尺度と対数尺度の散布図を見比べる。


In [ ]:
df['log_density'] = np.log(df['population_density_per_km2'])
df['log_inhabitable_density'] = np.log(df['inhabitable_density_per_km2'])
fig, ax = plt.subplots()
ax.scatter(df['log_density'], df['owned_home_floor_area_m2'],
           color='#777777', s=30, alpha=0.75)
annotate_prefectures(ax, df, 'log_density', 'owned_home_floor_area_m2')
ax.set(xlabel='Log population density (natural log)',
       ylabel='Owner-occupied floor area (m²)')
ax.xaxis.set_major_formatter(StrMethodFormatter('{x:.1f}'))
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.0f}'))
plt.tight_layout()
plt.show()


対数に変換した図では、低密度の県どうしの位置関係も読み取りやすくなる。全体として右下がりの関係がある一方、人口密度が近い県でも床面積には違いがある。例えば、石川と富山の人口密度はそれぞれ262.3人/km²と234.7人/km²だが、持ち家の床面積は富山のほうが15.7 m²広くなっている。


## 5. 単回帰と残差の復習

持ち家の床面積を被説明変数 $y_i$、人口密度の自然対数を説明変数 $x_i$ とする。説明変数が一つの単回帰モデルは

$$y_i=\beta_0+\beta_1x_i+u_i$$

である。$i$ は都道府県の番号、$\beta_0$ は切片、$\beta_1$ は傾き、$u_i$ はモデルの誤差を表す。

データから求めた係数を $\hat\beta_0,\hat\beta_1$ と書くと、当てはめ値と残差は

$$\hat y_i=\hat\beta_0+\hat\beta_1x_i,\qquad e_i=y_i-\hat y_i$$

である。ハットは推定値を表す。通常最小二乗法（Ordinary Least Squares, OLS）は、47県の残差二乗和 $\sum_i e_i^2$ を最小にする係数を選ぶ。残差は、未知の誤差 $u_i$ と異なり、データと当てはめ値から計算できる。

次のコードでは `sm.add_constant()` で切片用の列を加え、`sm.OLS(y, X).fit()` で係数を求める。


In [ ]:
x = df['log_density']
y = df['owned_home_floor_area_m2']
X = sm.add_constant(df[['log_density']])
result = sm.OLS(y, X).fit()
b0 = result.params['const']
b1 = result.params['log_density']
df['fitted'] = result.fittedvalues
df['residual'] = result.resid

print(f'回帰式：持ち家の床面積 = {b0:.2f} − {abs(b1):.2f} × ln(人口密度)')
print(f'決定係数 R² = {result.rsquared:.3f}')


In [ ]:
grid = np.linspace(x.min(), x.max(), 100)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(x, y, s=25, color='#777777', alpha=0.65)
ax.plot(grid, b0 + b1 * grid, color='#a31b2a', label='OLS line')
for name in labels:
    row = df.loc[df['prefecture'].eq(name)].iloc[0]
    ax.vlines(row['log_density'], row['fitted'], row['owned_home_floor_area_m2'],
              color=colors[name], linewidth=2)
annotate_prefectures(ax, df, 'log_density', 'owned_home_floor_area_m2')
ax.set(xlabel='Log population density (natural log)',
       ylabel='Owner-occupied floor area (m²)')
ax.xaxis.set_major_formatter(StrMethodFormatter('{x:.1f}'))
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.0f}'))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


### 石川県の当てはめ値と残差

図の縦線は、実際の面積と回帰直線の差を表す。石川県の当てはめ値は約130.8 m²、実際の面積は151.8 m²なので、残差は約 $151.8-130.8=+21.0$ m²である。正の残差は、実際の面積が当てはめ値より大きいことを表す。


In [ ]:
ishikawa = df.loc[df['prefecture'].eq('石川県')].iloc[0]
print(f"石川県の実際の面積：{ishikawa['owned_home_floor_area_m2']:.1f} m²")
print(f"回帰式による当てはめ値：{ishikawa['fitted']:.1f} m²")
print(f"残差：{ishikawa['residual']:+.1f} m²")


## 6. 回帰係数と決定係数の読み方

説明変数が人口密度の自然対数なので、人口密度が2倍違う県の回帰直線上の面積の差は $\hat\beta_1\ln 2$、約−7.8 m²である。

決定係数 $R^2$ は、この47県の持ち家面積のばらつきを直線がどの程度説明しているかを表す。

$$R^2=1-\frac{\sum_i e_i^2}{\sum_i(y_i-\bar y)^2}\approx0.451.$$

$\bar y$ は47県の平均面積である。この直線は面積のばらつきの約45％を説明している。各県の面積を45％の正確さで予測できるという意味ではない。

県平均どうしの関係には、世帯構成や地価などの違いも含まれる。この係数は人口密度の因果効果を測った値ではなく、個々の家計にもそのまま当てはまるわけではない。


## 7. 残差の広がり

残差の図では、回帰直線からのずれを0を基準に示す。正の値は直線より上、負の値は直線より下にある県である。

このデータでは、人口密度が低い側で残差の上下への広がりが大きくなっている。富山や石川には大きな正の残差が見られるが、人口密度が低くても負の残差をもつ県もある。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(df['log_density'], df['residual'], s=30, color='#777777', alpha=0.75)
ax.axhline(0, color='black', linewidth=1)
annotate_prefectures(ax, df, 'log_density', 'residual')
ax.set(xlabel='Log population density (natural log)', ylabel='Residual (m²)')
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.0f}'))
plt.tight_layout()
plt.show()

residual_table = df[['prefecture', 'owned_home_floor_area_m2', 'fitted', 'residual']].copy()
residual_table['abs_residual'] = residual_table['residual'].abs()
display(residual_table.nlargest(5, 'abs_residual')
        .drop(columns='abs_residual')
        .rename(columns={'prefecture':'都道府県', 'owned_home_floor_area_m2':'実際（m²）',
                         'fitted':'当てはめ値（m²）', 'residual':'残差（m²）'})
        .style.format({c:'{:.1f}' for c in ['実際（m²）','当てはめ値（m²）','残差（m²）']}))


**不均一分散**とは、説明変数の値によって、モデルの誤差の分散が異なることである。残差の広がりはその手掛かりになるが、図で見える残差は未知の誤差そのものではない。平均の関係式のずれや、少数の県の影響も考えられる。

したがって、この図からは「低密度の県で直線からのずれが大きい」と読み取れるが、その原因を不均一分散と断定することはできない。不均一分散を扱う回では、平均の関係を決めた数値実験で、誤差の分散だけを変えた場合を扱う。


## 8. 重回帰分析の導入

住宅の広さには、人口密度に加えて世帯人数や住宅の種類なども関係すると考えられる。複数の要因を一つの式で扱う方法が**重回帰分析**である。説明変数を二つ以上使い、各説明変数と被説明変数との関係を調べる。

例えば、人口密度と世帯人数を説明変数にすると、世帯人数を同じとした場合の人口密度と住宅面積との関係を表せる。人口密度だけを使う単回帰では、世帯人数などの違いも含めて県どうしを比較している。説明変数を加えると、係数が表す比較の条件も変わる。

以下では、手元のデータにある**二つの人口密度**を使って重回帰を計算する。この組合せは重回帰の式を学ぶ例であり、よく似た説明変数を使うときに生じる多重共線性の例でもある。


### 8.1. 二つの人口密度の定義

人口密度は「人口 ÷ 面積」だが、分母には総面積のほかに**可住地面積**も使われる。可住地面積は、総面積から林野面積と主要湖沼面積を引いたものである。総面積はこの統計の定義に従い、北方地域・竹島を除く。

同じ人口をより小さな面積で割るため、可住地面積当たりの密度は総面積当たりの密度以上になる。可住地面積の割合が小さいほど、可住地面積当たりの密度と総面積当たりの密度の比は大きくなる。


In [ ]:
density_table = (df.set_index('prefecture')
                 .loc[['東京都', '石川県', '富山県'],
                      ['population_density_per_km2', 'inhabitable_density_per_km2']]
                 .rename(columns={
                     'population_density_per_km2': '総面積当たり（人/km²）',
                     'inhabitable_density_per_km2': '可住地面積当たり（人/km²）'}))
display(density_table.style.format('{:,.1f}'))


二つの密度をそれぞれ自然対数に変換し、散布図に示す。横軸は総面積当たり、縦軸は可住地面積当たりである。相関係数は約0.954で、総面積当たりの密度が高い県は、可住地面積当たりの密度も高い傾向がある。


In [ ]:
correlation = df[['log_density', 'log_inhabitable_density']].corr().iloc[0, 1]
print(f'二つの対数人口密度の相関係数：{correlation:.3f}')
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(df['log_density'], df['log_inhabitable_density'], s=30,
           color='#777777', alpha=0.75)
annotate_prefectures(ax, df, 'log_density', 'log_inhabitable_density')
ax.set(xlabel='Log density: total area', ylabel='Log density: inhabitable area')
ax.xaxis.set_major_formatter(StrMethodFormatter('{x:.1f}'))
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.1f}'))
plt.tight_layout()
plt.show()


### 8.2. 重回帰モデルと当てはめ値

被説明変数は、引き続き持ち家の床面積 $y_i$ である。二つの説明変数を $x_{1i}$、$x_{2i}$ と書くと、重回帰モデルは

$$y_i=\beta_0+\beta_1x_{1i}+\beta_2x_{2i}+u_i$$

となる。添字の1と2は説明変数の種類、$i$ は都道府県を表す。

| 記号 | この分析での意味 |
|---|---|
| yᵢ | 持ち家の床面積（m²） |
| x₁ᵢ | 総面積当たりの人口密度の自然対数 |
| x₂ᵢ | 可住地面積当たりの人口密度の自然対数 |
| β₀ | 切片 |
| β₁, β₂ | それぞれの説明変数に掛かる回帰係数 |
| uᵢ | この式では説明しきれない違いを表す誤差 |

単回帰の式に、二つ目の説明変数の項 $\beta_2x_{2i}$ が加わっている。係数をデータから推定すると、当てはめ値は

$$\hat y_i=\hat\beta_0+\hat\beta_1x_{1i}+\hat\beta_2x_{2i}$$

である。$\hat\beta$ のハットは推定値であることを表す。一つの回帰式の中で、二つの説明変数を同時に使う。


### 8.3. ほかの説明変数を一定にした比較

重回帰の係数 $\hat\beta_1$ は、**$x_2$ を同じ値に保ち、$x_1$ だけを1増やしたときの当てはめ値の差**である。同じ $x_2$ を代入した二つの式を引くと、

$$\begin{aligned}
\hat y(x_1+1,x_2)-\hat y(x_1,x_2)
&=[\hat\beta_0+\hat\beta_1(x_1+1)+\hat\beta_2x_2]
\\
&\quad -[\hat\beta_0+\hat\beta_1x_1+\hat\beta_2x_2]\\
&=\hat\beta_1
\end{aligned}$$

となる。$\hat y(x_1,x_2)$ は、指定した二つの値を回帰式に代入した当てはめ値である。同様に、$\hat\beta_2$ は $x_1$ を一定にして $x_2$ を1増やしたときの差である。

この例の $x_1$ は人口密度そのものではなく、その自然対数である。**可住地密度を一定にして総面積密度が2倍違う場合**には、$x_1$ の差が $\ln 2$ になるので、当てはめ値の差は $\hat\beta_1\ln 2$ である。

「一定にする」とは、回帰式に同じ値を代入して比較することである。実際のデータから、もう一方の密度が完全に同じ県を探して二組だけで計算するわけではない。ただし、一方だけが違う県がほとんどなければ、その比較をデータから精度よく推定することは難しくなる。

また、二つの密度を使うだけで世帯人数や地価まで同じになるわけではない。この重回帰の係数も、そのまま人口密度の因果効果とは解釈できない。


### 8.4. 重回帰の最小二乗法

重回帰でも、残差は「実際の値 − 当てはめ値」である。OLS は、次の残差二乗和を最も小さくする三つの係数 $a,b_1,b_2$ を**同時に**選ぶ。

$$S(a,b_1,b_2)=\sum_{i=1}^{47}(y_i-a-b_1x_{1i}-b_2x_{2i})^2.$$

単回帰を二回行って、それぞれの係数を一つの式に並べる計算とは異なる。二つの説明変数を含む式全体の当てはまりを調べる。

コードでは `df[['log_density', 'log_inhabitable_density']]` で二列を選び、`sm.add_constant()` で切片用の列を加える。`sm.OLS(y, X_two).fit()` が係数を求める処理である。


In [ ]:
# 総面積密度の単回帰 result は第5節で計算済み。
# 比較のため、可住地密度だけを使う単回帰も計算する。
X_inhabitable = sm.add_constant(df[['log_inhabitable_density']])
result_inhabitable = sm.OLS(y, X_inhabitable).fit()

# 二つの説明変数を同じ式に入れる重回帰。
X_two = sm.add_constant(df[['log_density', 'log_inhabitable_density']])
result_two = sm.OLS(y, X_two).fit()
print('重回帰の式（係数は小数第2位まで表示）：')
print(f"床面積の当てはめ値 = {result_two.params['const']:.2f} "
      f"{result_two.params['log_density']:+.2f} × ln(総面積密度) "
      f"{result_two.params['log_inhabitable_density']:+.2f} × ln(可住地密度)")


### 8.5. 単回帰と重回帰の推定結果

三つの回帰を同じ47県のデータで比べる。**標準誤差**（SE）は、設定した統計モデルの下で、係数の推定値がどのくらいばらつくかを推定した尺度である。係数そのものの大きさや、床面積の残差とは区別する。

以下の通常SEは、各県の誤差が独立で分散が一定という仮定に基づく。住宅・土地統計調査の標本設計や県どうしの誤差の関連を反映した値ではない。第7節の残差の図からは分散が一定か検討が必要なので、ここでは同じ計算法で得たSEとして比較する。

決定係数 $R^2$ は当てはまりの指標である。同じ被説明変数・同じ観測を使う切片つきの OLS では、説明変数を追加しても $R^2$ は下がらない。追加した変数の係数を0にすれば元の式を使えるため、残差二乗和が増えることはないからである。**調整済み決定係数**は説明変数の数も考慮した指標で、変数を追加すると下がることもある。


In [ ]:
models = {'総面積密度の単回帰': result,
          '可住地密度の単回帰': result_inhabitable,
          '二つの密度の重回帰': result_two}
comparison = pd.DataFrame({name: {
    '総面積密度の対数の係数': model.params.get('log_density', np.nan),
    'その通常SE（総面積）': model.bse.get('log_density', np.nan),
    '可住地密度の対数の係数': model.params.get('log_inhabitable_density', np.nan),
    'その通常SE（可住地）': model.bse.get('log_inhabitable_density', np.nan),
    '決定係数 R²': model.rsquared,
    '調整済み決定係数': model.rsquared_adj,
} for name, model in models.items()})
display(comparison.style.format('{:.2f}', na_rep='—')
        .format('{:.3f}', subset=pd.IndexSlice[['決定係数 R²', '調整済み決定係数'], :]))


総面積密度の係数は、単回帰の約−11.24から重回帰の約−3.05に変わる。単回帰は可住地密度の違いも含む比較、重回帰は可住地密度を一定にした比較である。比較の条件が異なるので、係数が変わったことだけを根拠に、一方の結果を誤りとは判断できない。

重回帰では、可住地密度が同じで総面積密度が2倍違う場合の当てはめ値の差は、約 $-3.05\times\ln 2\approx-2.1$ m²である。単回帰の約−7.8 m²とは、同じ条件での比較ではないことに注意する。

二つの密度を使った重回帰の $R^2$ は約0.475で、可住地密度だけの単回帰の約0.472からわずかに増えている。一方、通常SEは総面積密度で1.85から6.12、可住地密度で2.40から8.11へ大きくなっている。


## 9. 多重共線性と係数の不確かさ

**多重共線性**は、説明変数どうしに強い線形関係があることをいう。今回の二つの対数人口密度には強い相関があり、一方を一定にしたまま他方が違う県の比較に使える情報が少なくなっている。

以下は仕組みを説明するための仮想的な数値例である。住宅の公表値とは分けて考え、切片を0に固定する。

### 9.1. 同じ説明変数を二つ入れた場合

すべての観測で $x_2=x_1=x$ なら、

$$2x+3x=1x+4x=5x$$

である。係数が「2と3」でも「1と4」でも、当てはめ値はまったく同じになる。一般に $b_1x+b_2x=(b_1+b_2)x$ なので、データから分かるのは係数の和である。二つの係数を別々に一意に決められない。これが**完全な多重共線性**の例である。


### 9.2. 説明変数がよく似ている場合

すべての観測で $x_2$ が $x_1$ に近い値をとる場合、次の二つの式を比べる。添字の A と B は式を区別するための記号である。

$$\hat y_A=2x_1+3x_2,\qquad \hat y_B=x_1+4x_2.$$

二つの予測値の差は

$$\hat y_B-\hat y_A=(x_1+4x_2)-(2x_1+3x_2)=x_2-x_1$$

である。$x_1=2.0$、$x_2=2.1$ なら、式Aは10.3、式Bは10.4となり、差は0.1である。個々の係数は1ずつ違うが、この組合せでは予測値が近くなる。

式全体の当てはまりが似ていると、「$x_1$ にどれだけ、$x_2$ にどれだけの係数を割り当てるか」をデータから明確に区別しにくくなる。


### 9.3. 観測値の小さな違いと係数の変化

まず、すべての観測値 $y$ が式Aに一致するデータを考える。二つの説明変数には完全な線形関係がないものとし、切片は0に固定する。OLS で求まる係数は2と3である。

各観測の $y$ を $x_2-x_1$ だけ変えると、

$$y+(x_2-x_1)=(2x_1+3x_2)+(x_2-x_1)=x_1+4x_2$$

となり、変更後の観測値には式Bが完全に当てはまる。$x_2$ と $x_1$ が近ければ $y$ の変更は小さいのに、係数は2から1、3から4へ変わる。次のコードでは、各観測値を0.1だけ増減させて、この計算を示す。


In [ ]:
# 仕組みを説明する仮想データ。住宅のデータ df とは別に作る。
toy_x1 = np.array([1.0, 2.0, 3.0, 4.0])
toy_x2 = np.array([1.1, 2.1, 2.9, 3.9])
toy_X = np.column_stack([toy_x1, toy_x2])
toy_y_A = 2 * toy_x1 + 3 * toy_x2
toy_change = toy_x2 - toy_x1
toy_y_B = toy_y_A + toy_change

# この数値例は切片を0に固定するため、add_constant は使わない。
toy_fit_A = sm.OLS(toy_y_A, toy_X).fit()
toy_fit_B = sm.OLS(toy_y_B, toy_X).fit()

display(pd.DataFrame({
    'x₁': toy_x1, 'x₂': toy_x2,
    '変更前のy（式A）': toy_y_A,
    'yの変更量': toy_change,
    '変更後のy（式B）': toy_y_B,
}, index=['観測1', '観測2', '観測3', '観測4']).style.format('{:.1f}'))
display(pd.DataFrame({
    '変更前のOLS': toy_fit_A.params,
    '変更後のOLS': toy_fit_B.params,
}, index=['x₁の係数', 'x₂の係数']).style.format('{:.1f}'))


この例は、小さな観測値の違いが個々の係数の大きな違いにつながり得ることを示している。観測値の変化を、係数の変化が分かりやすい形に設定している。どのような小さな変更でも、必ず同じ大きさで係数が変わるという意味ではない。

実際の二つの対数人口密度にも完全な線形関係はなく、係数は一意に求められる。ただし、強い線形関係があるため、個々の係数を精度よく求めにくくなっている。先ほどの表で両方のSEが大きくなったことは、この問題に対応している。


### 9.4. 説明変数の組合せと予測値

二つの式の予測値が近いかどうかは、説明変数の組合せにも依存する。$x_1=2.0$ のまま $x_2$ を2.1から5.0に変えると、式Aと式Bの差は次のようになる。

| x₁ | x₂ | 式Aの予測値 | 式Bの予測値 | 予測値の差（B − A） |
|---:|---:|---:|---:|---:|
| 2.0 | 2.1 | 10.3 | 10.4 | 0.1 |
| 2.0 | 5.0 | 19.0 | 22.0 | 3.0 |

観測された $x_1$ と $x_2$ が近い組合せでは予測値も近くなるが、$x_1$ だけを固定して $x_2$ を大きく変えると予測値の差が広がる。元のデータにはない組合せに対しても予測が安定しているとは限らない。

住宅の例でも、総面積密度と可住地密度は一緒に高くなる傾向がある。一方を一定にした比較を読むときには、その比較に近い組合せが実際のデータに十分あるかを考える必要がある。
